In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] ='0'
from huggingface_hub import notebook_login

- log in Hugging Face 

In [2]:
notebook_login()

# Phase 1: System Design & Stack Selection

- __Language & Framework:__ Python with LangChain. It provides a strong set of core components for building both RAG systems and tool-based agents.
- __Data Ingestion:__ WebBaseLoader for the SAGES and AME HTML links, and PyPDFLoader for the ERAS booklet.
- __Chunking Strategy:__ RecursiveCharacterTextSplitter. Since surgical steps and guidelines are highly sequential, overlapping chunks (e.g., 1000 tokens with a 200-token overlap) will help maintain context across operative steps.
- __Vector Store:__ ChromaDB or FAISS. Both are lightweight, run locally, and require zero external infrastructure, which is perfect for a take-home CLI tool.
- __Agent Paradigm:__ A Tool-Calling Agent. We will wrap your RAG pipeline into a "Clinical Knowledge Retriever" tool. We will also implement Structured Output using Pydantic, requiring the LLM to format its response with specific fields (e.g., current_step, next_action, safety_warnings) to satisfy the agent capability requirement.

### Here is a clean Python script using LangChain to ingest the web and PDF sources, along with a strategic chunking configuration.

# Phase 2: Data Ingestion & Chunking Strategy

In [3]:
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


### __Ingests clinical guidelines from URLs and PDFs, then chunks them for optimal Vector Store retrieval.__

## 1. Define our source documents

In [4]:
sages_url = "https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/"
ame_url = "https://ales.amegroups.org/article/view/5766/html"
eras_pdf_url = "https://www.urmc.rochester.edu/getmedia/c6bc9e17-c349-436c-926e-bf4f4e498d8d/ERAS-Cholecystectomy-Booklet.pdf"

### Load HTML Web Documents (SAGES & AME)

At a high level, *WebBaseLoader* in LangChain is a tool that handles the process of pulling HTML from a URL and turning it into a clean, usable document object.

I used it to load sources like the SAGES guidelines and the AME Operative Technique article. While I could have written a custom BeautifulSoup script, the advantage of *WebBaseLoader* is that it standardizes the output for the rest of the LangChain pipeline.

Most importantly, it automatically keeps the source URL as part of the document’s metadata. In a clinical workflow, knowing where the information comes from is important; if the agent suggests a specific surgical technique, we need to link that recommendation back to the exact SAGES source.

In [5]:
web_loader = WebBaseLoader(web_paths=[sages_url, ame_url])
web_docs = web_loader.load()

In [ ]:
web_docs[0]

### Load PDF Document (ERAS)

Similar to the *WebBaseLoader*, *PyPDFLoader* is LangChain’s built-in tool for working with PDF files. I used it to process the ERAS Cholecystectomy booklet.

The main advantage is how it handles document structure. Instead of loading the entire PDF as one large block of text, it automatically splits the content page by page and includes the page number in the metadata for each section.

This is very useful during retrieval. If the agent pulls a safety protocol from the ERAS booklet, we can immediately see which page it came from. It makes debugging the pipeline much easier and helps keep the model’s responses clearly connected to the source.

In [6]:
pdf_loader = PyPDFLoader(eras_pdf_url)
pdf_docs = pdf_loader.load()

In [ ]:
pdf_docs

In [7]:
len(pdf_docs)

12

### Combine all raw documents

In [8]:
all_raw_docs = web_docs + pdf_docs
print(f"Successfully loaded {len(all_raw_docs)} raw documents.")

Successfully loaded 14 raw documents.


### Chunking Strategy

Chunking means taking a large document and splitting it into smaller sections before storing it in a vector database.

We do this for two main reasons. First, LLMs have limits on how much text they can process at once, so we can’t include an entire textbook in a single prompt.

More importantly, it improves search accuracy. If I embed a 50-page surgical manual as one vector, its mathematical meaning becomes too averaged out. But if I split it into smaller sections, the model can retrieve the exact part that’s relevant.

For example, if a user asks about Calot’s triangle, we want the system to return the specific paragraph that explains it, not the entire manual. Chunking makes that retrieval much more precise.

__I chose a chunk_size of 1000 tokens with a chunk_overlap of 200. This is a standard "Goldilocks" zone for medical texts; large enough to capture an entire concept (like the criteria for the Critical View of Safety) but overlapping enough so that continuous operative steps aren't artificially severed.__

*RecursiveCharacterTextSplitter* is a specific chunking method in LangChain, and it’s one of the most effective options for working with standard text. Instead of splitting the text at fixed intervals, like every 500 words, which can break sentences or surgical steps, it uses a step-by-step approach.

It first tries to split by double newlines to keep paragraphs intact. If a section is still too large, it then splits by single newlines to preserve sentence structure, and continues in that way.

I chose this method because clinical guidelines are highly structured, with headings, bullet points, and ordered steps. The key advantage is that it follows the natural structure of the human language. This helps ensure that important details, like safety warnings, stay connected to the surgical steps they relate to.

- The __chunk_size__ is simply the maximum number of characters in each text segment. For this pipeline, I set it to 1000 characters. In practice, this works well for medical text; it’s large enough to cover an entire concept, like the full criteria for the Critical View of Safety, but still small enough to keep retrieval accurate.

- The __chunk_overlap__ defines how much text is shared between neighboring segments. I set it to 200 characters, meaning the last part of one segment is repeated at the start of the next. This is especially important for surgical workflows, where steps follow a clear sequence. If one step ends at the boundary and the next begins in a new segment, the overlap helps preserve that connection so the model doesn’t lose important context.

In [9]:
# Using RecursiveCharacterTextSplitter to split by paragraphs/sentences to preserve clinical context boundaries.

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True, # Helps track where in the document the chunk came from
        separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

chunked_docs = text_splitter.split_documents(all_raw_docs)
    
print(f"Split documents into {len(chunked_docs)} manageable chunks.")

Split documents into 274 manageable chunks.


In [12]:
print("\n--- Preview of Chunk 1 ---")
print(f"Source: {chunked_docs[2].metadata.get('source')}")
print(f"Content: {chunked_docs[2].page_content[:300]}...\n")


--- Preview of Chunk 1 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content: Member Spotlight
Give the Gift of SAGES Membership


Patients

Join the SAGES Patient Partner Network (PPN)
Patient Information Brochures
Healthy Sooner – Patient Information for Minimally Invasive Surgery
Choosing Wisely – An Initiative of the ABIM Foundation
All in the Recovery: Colorectal Cancer ...



__Unified Pipeline: It seamlessly handles both standard web scraping and PDF parsing in one go.__

__Metadata Preservation: The LangChain loaders automatically attach the source URLs to the metadata of each chunk. This is critical for RAG, as my agent will eventually need to cite exactly which document it pulled the surgical step from.__

Now that we have our text neatly divided into overlapping clinical chunks, we need to embed them into a Vector Store so our model can query them.

# Phase 3: The RAG Pipeline (Embedding & Retrieval)

__Reason for Embedding:__
You can’t just pass raw text into an AI system and expect it to understand how clinical concepts relate to each other. Embedding acts as a translation layer. It takes our human-readable sections of surgical guidelines and converts them into numerical vectors.

We do this because a simple keyword search isn’t reliable enough for medical data. For example, if a user asks about ‘removing the gallbladder,’ but the document uses the term ‘cholecystectomy,’ a keyword search might miss it.

With embeddings, the model captures the meaning behind the words, so those two terms are represented very closely in vector space. This is what allows the system to retrieve the most relevant information in a smart and flexible way.

Standard top-k retrieval might pull four chunks that all describe the exact same sentence from different angles. MMR fetches a larger pool of chunks and then selects the most diverse set, ensuring the LLM gets a broader context of the surgical procedure (e.g., pulling a chunk about Calot's triangle dissection and a chunk about the safety risks).

For the embedding model, *BAAI/bge-small-en-v1.5* (via Hugging Face) is the best lightweight embedding model for retrieval tasks. It runs fast and has excellent semantic understanding of __medical texts__. For the vector database, ChromaDB is perfect because it stores the database locally as a file, requiring no complex server setup for the live demo.

## 3.1 Initializing Local Embeddings
To maintain strict data privacy suitable for hospital environments, we utilize a local, open-weights embedding model (`BAAI/bge-small-en-v1.5`). We are accelerating this process using GPU compute.

In [10]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

__What & Why is HuggingFaceEmbeddings?__

It’s basically LangChain’s connection to the Hugging Face ecosystem. Instead of writing custom PyTorch code to load a model, tokenize text, pool hidden states, and manage device placement, HuggingFaceEmbeddings wraps all of that into a simple, easy-to-use class.

I chose it here because it supports running everything locally. By setting the device to ‘cuda,’ I could run the embedding step directly on the V100 GPU in our HPC environment, keeping the pipeline private and very fast.

__What & Why is BAAI/bge-small-en-v1.5?__

BGE-small is an open-weight embedding model developed by the Beijing Academy of Artificial Intelligence. I chose it because it performs very strongly on benchmarks like MTEB (Massive Text Embedding Benchmark).

The key advantage of this clinical retrieval task is efficiency. It’s a smaller model, producing 384-dimensional vectors compared to the larger 1536-dimensional vectors from models like OpenAI’s.

Since I’m running everything locally, I don’t need a large model to achieve accurate semantic search. This model provides a strong balance. It captures detailed medical meaning while remaining fast enough that the user experiences near real-time performance.

__Additional Note__

BGE-small is a general-purpose embedding model, rather than a medical-specific model like ClinicalBERT or BioBERT, which are trained mainly on clinical notes and PubMed articles. I chose it intentionally because the field has evolved quite a bit.

Newer general models like BGE are trained on very large and diverse datasets, including a significant amount of scientific and medical literature. Even though the data isn’t only medical, the overall amount of medical content it has seen is much larger than what older specialized models were trained on.

Another key point is how it’s trained. BGE uses contrastive learning on pairs of text, such as questions and answers. This makes it especially good at connecting clinical terminology in guidelines with the natural language a user might type.

For a lightweight pipeline running on a single GPU, BGE-small offers a strong balance. It captures medical meaning accurately while staying fast enough for real-time use, without relying on a large, resource-heavy model.

In [11]:
print("Initializing local HuggingFace embeddings...")

# Initialize Local Embeddings
# Pushing the model to the GPU for near-instantaneous embedding generation
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda:0'} 
)

print("Embeddings loaded successfully on GPU.")

Initializing local HuggingFace embeddings...
Embeddings loaded successfully on GPU.


## 3.2 Building the Vector Database

__Purpose of building the Vector Database:__

If embeddings are like GPS coordinates for our clinical data, then the vector database is the map.

After converting sections of the SAGES and ERAS guidelines into numerical vectors, we need an efficient system to store and search them. When a user asks a question, we convert that question into a vector as well, and the database compares it to all stored vectors using cosine similarity, which measures how close they are in meaning.

It then returns the most relevant sections based on that similarity. Without a vector database, the retrieval step in RAG wouldn’t be possible.

__Why did we choose langchain_community.vectorstores.Chroma?__

I chose Chroma because it’s lightweight and works well for a privacy-focused setup.

In a clinical setting, I wanted to avoid cloud-based vector databases, since sending hospital data to external servers conflicts with a local, secure design. At the same time, I didn’t want the added complexity of setting up a full-scale database system just for a demo.

Chroma runs locally within the Python environment, and its persist_directory feature allows me to save the database directly to disk. I can build the index once and then load it quickly for a live demo, without waiting for it to rebuild each time.

__ChromaDB__ is an open-source vector database. If embeddings are like coordinates, Chroma is the system that stores and organizes them.

I chose it because it’s lightweight and easy to work with. In a cloud production setup, you might use something like AWS OpenSearch. But for a clinical, local setup, and especially for a live demo, Chroma works really well.

The key advantage is that it runs locally and can save data directly to a disk folder. I didn’t need to set up containers or rely on external APIs. I could embed the SAGES and ERAS guidelines once, save them, and then load them quickly without network delays or data privacy concerns.

We use ChromaDB to store our chunked clinical guidelines locally. The database is persisted to disk so it can be instantly reloaded in future sessions without re-embedding.

In [12]:
persist_dir = "./chroma_db_clinical"
print("Building and persisting local Vector Store. This may take a moment...")

vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

print(f"Successfully embedded chunks into local ChromaDB at {persist_dir}.")

Building and persisting local Vector Store. This may take a moment...
Successfully embedded chunks into local ChromaDB at ./chroma_db_clinical.


## 3.3 Configuring the Clinical Retriever

__Retriever__

The vector database mainly stores numerical data; it doesn’t actively perform the search by itself. The retriever is the component that handles the search process.

We need this step because the LLM and the database work in different formats. When a user asks a question like, ‘What are the risks of dissecting Calot’s triangle?’, the retriever takes that text, converts it into a vector using the embedding model, and runs a similarity search against ChromaDB.

It then converts the results back into readable sections of text. In that sense, it connects the user’s question with the relevant information stored in the database.

__Metric of mathematical distance:__

The equation for Cosine Similarity is:

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|}$$

In this pipeline, using BGE-small embeddings and ChromaDB, the distance measure we rely on is cosine similarity.

When the embedding model converts clinical guidelines into a 384-dimensional vector space, we’re not focused on straight-line distance like Euclidean distance. Instead, we compare the angle between vectors. Mathematically, this is the dot product of the query and document vectors, divided by the product of their lengths.

We use cosine similarity because the length of a vector is affected by how long the text is. For example, a long paragraph about gallbladder removal and a short summary may have very different lengths, but they can still represent the same meaning. By focusing on the angle, we capture the meaning of the text while ignoring differences in text length.

Standard retrieval often pulls highly redundant chunks. To ensure our agent receives a comprehensive view of the surgical step and associated safety risks, we configure the retriever to use Maximal Marginal Relevance (MMR).

__Maximal Marginal Relevance (MMR)__

Standard vector search focuses on the highest similarity score. The issue is that if I ask about Calot’s triangle, it might return several paragraphs from the same document that repeat the same idea in slightly different ways.

Maximal Marginal Relevance, or MMR, addresses this by balancing relevance with diversity. It selects results that are closely related to the query, but it penalizes them if they are too similar to the chunks it has already selected.

I used this approach for the clinical retriever because surgeons need comprehensive context, not repeated information. For example, if they ask about a dissection step, MMR can return one section on anatomy, another on the tools involved, and another that explains the safety risks. This gives the LLM a more complete set of information to generate a high-quality answer.

__Math Behind MMR:__

$$MMR = \arg\max_{D_i \in R \setminus S} \left[ \lambda \cdot Sim_1(D_i, Q) - (1 - \lambda) \cdot \max_{D_j \in S} Sim_2(D_i, D_j) \right]$$

The math behind MMR is an optimization equation that balances two metrics: relevance and diversity. It uses a penalty term and a weighting parameter, usually called λ.

When the algorithm evaluates a new candidate chunk, it calculates a score in two parts. The first part calculates the similarity between the document and the user's query; we want this to be high. The second part applies a penalty by measuring the maximum similarity between this new candidate and the chunks already selected.

If we set λ to 1, the penalty is removed, and it behaves like standard semantic search, which can return identical paragraphs. But when we lower λ, for example, to 0.5, the score is reduced if a chunk is too similar to the ones already selected.

In simple terms, the algorithm selects results that are both relevant to the query and different from each other, giving the model a more complete and useful context.

__Used Parameter:__

- `k`: sets how many text sections we pass to the LLM. We need this because the model has a limited context window, so we have to be careful about how much information we include. If we send too much text, the model can lose focus or miss important details. I chose `k = 4` because, with a chunk size of about 1000 tokens, it provides around 4,000 tokens of context. In my experience, this works well—it gives enough detail for complex clinical questions while still leaving room for the prompt and the model’s response within the 8k limit.

- `fetch_k`: The `fetch_k` parameter works together with MMR. It defines how many candidate text sections we consider before selecting the final set. If we only looked at 4 text sections from the start, MMR wouldn’t have enough options to choose from if those results were too similar. By setting `fetch_k = 15`, we first identify the top 15 most relevant sections. Then MMR reviews those and selects the best 4 while keeping the results diverse. I chose 15 because it’s a practical and effective setting; it gives a broad enough set to capture different aspects of a surgical topic, while still being fast enough to keep the system responsive during a live demo.

In [13]:
# Configure the Retriever with MMR
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,         # Return exactly 4 distinct chunks to the LLM
        "fetch_k": 15   # Initially fetch 15 similar chunks to evaluate for diversity
    }
)

print("Retriever configured with MMR (k=4, fetch_k=15).")

Retriever configured with MMR (k=4, fetch_k=15).


When the user asks a question, first, it pushes the user's question through the `BGE-small` embedding model to turn it into a vector. Then, the __Retriever__ steps in. It scans the ChromaDB vector database and calculates the mathematical distance between the question's vector and the vectors of all our document chunks. I specifically configured it to use __MMR__ to retrieve the top 4 most relevant, yet diverse, chunks of clinical guidelines.

## 3.4 Testing the Retrieval Pipeline
Let's verify the pipeline retrieves accurate and diverse context for a critical safety concept in laparoscopic cholecystectomy.

In [15]:
# Test query relevant to the clinical guidelines
test_query = "What is the Critical View of Safety (CVS)?"
print(f"Querying Vector Store: '{test_query}'\n")

retrieved_docs = retriever.invoke(test_query)

# Display the source and content of the top retrieved chunks
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content snippet: {doc.page_content[:300]}...\n")

Querying Vector Store: 'What is the Critical View of Safety (CVS)?'

--- Retrieved Chunk 1 ---
Source: https://ales.amegroups.org/article/view/5766/html
Content snippet: Step 2: Establishing the critical view of safety...

--- Retrieved Chunk 2 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: GUIDELINE RECOMMENDATIONS:
Question 1: Should the critical view of safety (CVS) versus other techniques (e.g. infundibular, top down, or intraoperative cholangiography) be used to mitigate the risk of bile duct injury during laparoscopic cholecystectomy?
Recommendation: In patients undergoing laparo...

--- Retrieved Chunk 3 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: Narrative synthesis: Forty-five full text articles identified by the search methodology were reviewed that included three systematic reviews.
Use of Critical View of Sa

# Phase 4: Agent Construction & Structured Reasoning

## 4.1 Loading the Inference Engine (vLLM)
Instead of relying on external APIs, we load Llama-3 8B directly into the V100 GPU's memory using vLLM. This ensures maximum privacy for clinical environments while providing high-throughput token generation.

__vLLM__ is an open-source inference algorithm designed to run large language models fast and efficiently. If I used a standard Hugging Face `pipeline`, it would be slower because traditional Transformers don’t manage memory very efficiently during text generation.

vLLM addresses this with a method called `PagedAttention`. It manages the model’s memory (the KV cache) in smaller segments, similar to how an operating system handles memory. This greatly reduces memory waste and improves performance.

I chose it because, in a clinical setting, the system needs to respond in real time during a procedure. vLLM provides that speed while maintaining accuracy.

- __High Performance:__ It makes better use of the GPU through techniques like continuous batching, which processes requests in real time and reduces waiting periods.
- __PagedAttention:__ This approach is based on how operating systems handle memory. It divides memory into smaller sections, which greatly reduces fragmentation in the KV cache and allows the model to handle larger batch sizes more efficiently.
- __Ease of Use:__ It integrates smoothly with Hugging Face models and offers straightforward Python APIs, an OpenAI-compatible server, and support for different types of hardware.
- __Wide Compatibility:__ It supports a wide range of popular models, like LLaMA, Mistral, and Mixtral, and includes features like tensor parallelism for efficient multi-GPU performance.

__What is PagedAttention in vLLM?__

Think about how an operating system manages RAM. In older LLM setups, like standard Hugging Face pipelines, the system tries to reserve memory for both the prompt and the generated output all at once, in one large continuous block. Since it doesn’t know how long the output will be, it often reserves more memory than needed, which leads to a lot of wasted space—sometimes over 50% of the available VRAM.

PagedAttention addresses this by following a similar idea to virtual memory in operating systems. Instead of requiring one large block, it divides memory into smaller, fixed-size sections called pages. As Llama-3 generates tokens step by step, memory is assigned page by page as needed. This greatly reduces memory waste.

This is the key idea behind why vLLM can deliver high speed and efficiency, even on a single V100 GPU.

__What is KV-cache in vLLM?__

The KV-cache, or Key-Value cache, is essentially the model’s short-term memory during text generation.

Since LLMs generate text step by step  (one word at a time), they would normally need to recalculate the attention matrices for the entire previous context each time they generate a new token. For example, with a 1000-token clinical context, recalculating all 1000 tokens just to produce the next one would be very slow and costly.

To address this, the model computes the ‘Key’ and ‘Value’ tensors once and stores them in the cache. When generating the next step—like the next part of a surgical procedure—it only needs to compute the new token and can reuse the stored context from the KV-cache.

The challenge is that this cache grows quickly, which is why vLLM uses PagedAttention to manage it efficiently and prevent the GPU from running out of memory during longer sequences.

__Why did you choose `meta-llama/Meta-Llama-3-8B-Instruct`?__

This choice was based on balancing hardware limits and model alignment. The 8-billion parameter size works well here—it fits smoothly within the 32GB of VRAM on a single V100 GPU, while still leaving space for the embedding model.

The key factor is the Instruct version. The base Llama-3 model mainly predicts the next token, so if you ask a question, it may just generate more questions instead of giving a clear answer. The Instruct model has already been trained with methods like Supervised Fine-Tuning and Direct Preference Optimization, so it knows how to follow system prompts and respond in a structured way.

I needed a model that could reliably follow the Pydantic format and produce clean JSON outputs every time, and the Instruct version does that without additional setup.

__Why set `max_new_tokens=1024`?__

That parameter acts as a safety limit. It defines the maximum number of tokens the model can generate for a single response.

I set it to 1024 to match our JSON structure. For complex questions, the model needs enough space to produce the full response, including the JSON fields, the primary answer, and any safety notes. 1024 tokens give more than enough room for a detailed clinical answer.

At the same time, it sets a clear upper limit on computation. If the model ever gets stuck or starts repeating itself, it will stop at 1024 tokens instead of continuing and using up the GPU.

__1024 tokens is around how many words?__

As a general rule of thumb for standard English text, 1 token is roughly equal to ¾ of a word (or 100 tokens = 75 words). Therefore, setting `max_new_tokens=1024` gives the model a limit of roughly 750 to 800 words.

__What is the temperature parameter?__

Temperature controls how much randomness the model uses when choosing the next word. The model assigns probabilities to many possible next words, and a higher temperature flattens those probabilities, allowing for more unexpected choices.

In my setup, I set the temperature to `0.0`. This makes the model always choose the most likely next word. For creative tasks, a higher temperature can be useful. But for a clinical assistant, especially when interpreting something like the Critical View of Safety, we want consistency and accuracy.

Setting the temperature to `0.0` ensures the model produces stable, reliable responses that stay closely aligned with the clinical context.

__Why set `gpu_memory_utilization = 0.8`?__

By default, vLLM tries to reserve most of the available VRAM to maximize its PagedAttention KV-cache size. If left unchanged, it can take over 90% of the GPU memory.

The challenge is that I’m also running the `BGE-small` embedding model on the same GPU. If vLLM uses all available memory, the embedding model will fail when processing a user query and crash the whole RAG pipeline.

By setting the memory usage to `0.8`, I’m dividing the GPU resources so that vLLM uses 80% of the VRAM, while the remaining 20% is kept available for the embedding model and system needs. This setup allows both models to run smoothly on a single node.

In [ ]:
from langchain_community.llms import VLLM

project_path = "/dartfs/rc/nosnapshots/V/VaickusL-nb/EDIT_Students/users/JiQing/LLM Project"

print("Loading vLLM directly into notebook memory. This will allocate GPU VRAM...")

# Initialize vLLM inline. 
# We limit gpu_memory_utilization to 0.8, so it leaves room for our Chroma embeddings.
llm = VLLM(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    trust_remote_code=True,  # Required for some modern HuggingFace models
    max_new_tokens=1024,
    temperature=0.0,         # Zero temperature for clinical accuracy
    vllm_kwargs={"gpu_memory_utilization": 0.8},
    download_dir = f"{project_path}/cache"
)

print("\nvLLM engine loaded successfully!")

By design, vLLM reserves almost all available GPU memory upfront (based on your *gpu_memory_utilization: 0.8* setting) to optimize the KV-cache.

When I re-run the above Cell in the Jupyter Notebook without properly shutting down the previous run, the old VLLM::EngineCore processes stay alive in the background, holding onto that 26.9GB of VRAM. When the new cell execution tries to start a new vLLM instance, it finds the GPU is full, fails to initialize, and throws that *Engine core initialization failed error*.

So I need to clear the GPU before running. In terminal:
*kill -9 1634243 1700459*

*1634243 1700459* are job IDs

## 4.2 Defining the Structured Output Schema
To satisfy the requirement for structured reasoning, we define a Pydantic schema. This ensures the LLM's response always breaks down the clinical scenario into exact, machine-readable fields (ex. Current Step, Next Action, Safety Warnings).

We will use *Optional* types and add a *query_intent* field. This forces the LLM to pause and categorize the question before generating the rest of the JSON.

__`SurgicalQAOutput(BaseModel)`:__

You can think of `BaseModel` as the core building block for how we define and control our data. It’s a class from the `pydantic` library that adds type checking and validation to our Python objects.

When our `SurgicalQAOutput` class is built on top of `BaseModel`, it turns into a strong validation layer. Its role is to make sure the data coming from the LLM matches exactly what we expect before the application uses it.

For example, if I define `safety_warnings` as a list of strings, but the model returns something incorrect—like a single number—BaseModel will catch the issue, raise a validation error, and prevent that data from reaching the UI or clinical dashboard.

In [15]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# Define a flexible, intent-driven structure
class SurgicalQAOutput(BaseModel):
    query_intent: str = Field(description="Classify the query into one of four categories: 'Surgical Step Understanding', 'Clinical Reasoning', 'Knowledge Retrieval', or 'Context-based'.")
    primary_answer: str = Field(description="A detailed explanation answering the core question, grounded ONLY in retrieved guidelines.")
    current_step: Optional[str] = Field(default=None, description="The current surgical step. ONLY output this if the query involves a specific point in the surgery. Otherwise, output null.")
    next_action: Optional[str] = Field(default=None, description="The immediate next step. ONLY output this if applicable to the query. Otherwise, output null.")
    safety_warnings: Optional[List[str]] = Field(default=None, description="Key risks or safety considerations. ONLY output if applicable to the query. Otherwise, output null.")

# Initialize the parser
output_parser = JsonOutputParser(pydantic_object=SurgicalQAOutput)

print("Flexible Pydantic schema and JSON parser initialized.")

Flexible Pydantic schema and JSON parser initialized.


## 4.3 Building the RAG Generation Pipeline
We now link our Phase 3 Retriever to our Phase 4 LLM. The prompt dynamically injects the retrieved clinical guidelines, the user's question, and the JSON formatting instructions.

In addition to structured output, the system uses a __Multi-step reasoning__ process. Early on, using a fixed schema led the model to generate incorrect surgical steps for general knowledge questions.

To address this, I designed an intent-aware prompt that guides the model through a step-by-step process. __Step 1:__ It reviews the user’s question and identifies the clinical intent. __Step 2:__ Decide which fields of the JSON structure are relevant based on that classification. __Step 3:__ It generates the final response.

This step-by-step approach helps prevent incorrect or misleading structured outputs.

__`PromptTemplate`:__

A `PromptTemplate` is a reusable way to structure how we communicate with the LLM.

Instead of building a long f-string every time a user asks a question, it lets us define the overall format once. It includes our main system instructions, like ‘You are an expert AI clinical assistant’, and then sets up clear input fields, such as `{context}` and `{question}`.

Here’s how it works: right before the model runs, the pipeline fills in those fields with the retrieved clinical guidelines into the `{context}` and the user’s question into the `{question}` slot. This keeps the prompt consistent every time and makes the model’s behavior much more predictable.

- __`partial_variables`:__ This is a very practical feature for managing prompt complexity. In our template, some variables change every time, like the user’s question and the retrieved context, while others stay the same, such as the JSON formatting rules generated by the Pydantic parser. The `partial_variables` argument lets us set those fixed formatting instructions in advance. By doing this, the JSON rules are already included in the prompt when it’s created. This simplifies the pipeline, because when a user asks a question, the system only needs to pass in the context and the question; the formatting rules are already part of the prompt.

- __`RunnablePassthrough`:__ This is a really useful feature in LangChain Expression Language. It helps manage how data moves through the pipeline. When a user enters a question, it starts as a simple string. But our `PromptTemplate` expects a dictionary with two keys: `context` and `question`. So we need to send that same input in two directions—one to the retriever to get relevant documents, and the other directly into the prompt as the user’s question. `RunnablePassthrough()` is designed for it. It takes the original input and forwards it unchanged into the pipeline. This allows us to run retrieval while keeping the original question intact, so both pieces come together correctly before being passed to the LLM.

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define the Intent-Aware Prompt
prompt = PromptTemplate(
    template="""You are an expert AI clinical assistant specializing in laparoscopic cholecystectomy. 
    Answer the user's question using ONLY the provided clinical context.
    
    CRITICAL INSTRUCTIONS:
    1. First, classify the user's question type (e.g., Knowledge Retrieval vs. Surgical Step).
    2. Provide the main answer in the 'primary_answer' field.
    3. ONLY fill out 'current_step', 'next_action', and 'safety_warnings' if the question implies a specific point in the surgical workflow (e.g., dissecting Calot's triangle). 
    4. If the question is a general definition or protocol (like ERAS guidelines or CVS), you MUST set 'current_step', 'next_action', and 'safety_warnings' to null.
    
    Clinical Context:
    {context}
    
    User Question: {question}
    
    {format_instructions}
    
    Output purely the JSON object without any markdown wrapping or additional text.
    """,
    input_variables=["question", "context"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

# Helper function to format the retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the LangChain Expression Language (LCEL) Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

print("Intent-aware RAG Pipeline linked and ready for inference.")

Intent-aware RAG Pipeline linked and ready for inference.


Retrieving the documents is just the first step; the system also needs to use them correctly without generating incorrect information. I designed the LangChain Expression Language chain pipeline to pass the retrieved content directly into the prompt.

LangChain Expression Language, or LCEL, may look a bit unusual at first because of the pipe syntax (`|`), but it’s actually a very structured, step-by-step data pipeline.

Here’s how a single query, like ‘What are the risks of dissecting Calot’s triangle?’, moves through the system, from the moment the user submits it to the final JSON output:

__Step 1: Input Split (Parallel Execution)__
When we call `rag_chain.invoke(user_query)`, the raw text enters the first block of the pipeline:
`{"context": retriever | format_docs, "question": RunnablePassthrough()}`.
The system immediately splits the data flow into two parallel paths:
- Question Path: `RunnablePassthrough()` takes the original text and passes it forward as the `question` variable.
- Context Path: The same text is sent to the `retriever` to fetch relevant documents.

__Step 2: Vector Search (Retrieval)__
This is where embedding happens. The retriever takes the input text and uses the `BGE-small` model to convert it into a 384-dimensional vector. It then sends that vector to ChromaDB, where Maximal Marginal Relevance (MMR) is applied.

Importantly, the database doesn’t return vectors to the LLM. Instead, it uses similarity scores to find the most relevant text sections/chunks and then returns the original, human-readable content linked to those sections/chunks. The `format_docs` function then combines those four sections/chunks into a single block of clinical context.

__Step 3: Prompt Injection__
At this stage, the pipeline combines both paths. We now have a dictionary that includes the `{context}`, which is the full text retrieved from ChromaDB, and the `{question}`.

This dictionary is then passed into the `PromptTemplate`. The template fills in these variables, along with the fixed Pydantic formatting instructions, to create the final prompt that is sent to the model.

__Step 4: LLM Generation__
The final formatted prompt is then sent to `vLLM`. The Llama-3 model processes the system instructions, the clinical context, and the user’s question. It computes attention weights and generates the response step by step, one token at a time.

Because of how the prompt is designed, the output is produced in a raw JSON format.

Because I clearly instruct the Llama-3 model to answer using only the provided clinical context, it relies on those retrieved guidelines as its main reference when reasoning through the {question} and completely bypasses its own pre-trained biases before generating the final JSON output.

__Step 5: Output Parsing (Final Validation Step)__
The raw string generated by vLLM is then sent to the `output_parser`. This is where Pydantic handles the validation.

The model may include extra conversational text or wrap the response in Markdown, like JSON tags. The parser removes that extra formatting, extracts the JSON content, and checks it against our Pydantic `BaseModel`.

It then converts the result into a clean, highly structured Python dictionary. This final, verified object is what gets displayed or passed to the clinical UI

By using LCEL, I don’t need to write custom Python if/else logic to manage all the transitions between steps. The pipe syntax moves data cleanly from the vector retrieval stage, into the prompt, then through the model on the GPU, and finally through the validator. This makes the entire RAG architecture easy to follow and ready for deployment.

## 4.4 Live Inference Test
Let's test the system with a complex, pseudo-visual reasoning question from the assignment prompt.

### Creating an assistant function

In [17]:
import json

def Clinical_Agent():
    print("=====================================================")
    print("⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized")
    print("Type 'exit' or 'quit' to end the session.")
    print("=====================================================\n")

    # Start an interactive CLI loop
    while True:
        # 1. Get user input
        user_input = input("\n🧑‍⚕️ Surgeon (You): ")
    
        # 2. Check for exit commands
        if user_input.lower() in ['exit', 'quit']:
            print("\nEnding session. Goodbye!")
            break
        
        # Skip empty inputs
        if not user_input.strip():
            continue
        
        print("\n🤖 AI Assistant is thinking and searching guidelines...")
    
        try:
            # 3. Invoke the RAG chain with the user's input
            result = rag_chain.invoke(user_input)
        
            # 4. Print the structured output beautifully
            print("\n--- Structured Clinical Output ---")
            print(json.dumps(result, indent=4))
        
        except Exception as e:
            print(f"\n❌ An error occurred: {e}")

### Question Category: Surgical Step Understanding

In [21]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the current step if the surgeon is dissecting Calot’s triangle?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The current step is the dissection of the hepatocystic triangle.",
    "current_step": "Dissection of the hepatocystic triangle",
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [22]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the next step after identifying the cystic duct and artery?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "After identifying the cystic duct and artery, the next step is to clip the cystic artery and divide it using hook scissors, taking care not to dislodge the proximal clips.",
    "current_step": "Step 3: Cystic artery is clipped and divided",
    "next_action": "Division of the cystic duct",
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Clinical Reasoning

In [23]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the key safety considerations during cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The key safety considerations during cholecystectomy include identifying the critical view of safety, maintaining a clear dissection plane, and avoiding excessive retraction. Additionally, surgeons should be aware of the risk of bile duct injury and take steps to minimize it, such as using a laparoscopic cholecystectomy technique that emphasizes the principles of safe cholecystectomy as highlighted by the Society of American Gastrointestinal and Endoscopic Surgeons (SAGES).",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [25]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the risks at the stage of cystic duct dissection?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The risks at the stage of cystic duct dissection are not explicitly mentioned in the provided clinical context. However, it is essential to identify and tape ligate the cystic duct prior to fundus-first dissection of the gallbladder to minimize the risk of bile duct injury.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Knowledge Retrieval

In [26]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the Critical View of Safety (CVS)?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "In patients undergoing laparoscopic cholecystectomy, the Critical View of Safety (CVS) is a technique used for anatomic identification of the cystic duct and artery.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [27]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the ERAS recommendations for cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The ERAS recommendations for cholecystectomy include a multidisciplinary approach to patient care, with a focus on minimizing postoperative complications and improving patient outcomes. This includes preoperative optimization, intraoperative techniques, and postoperative care protocols.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Context-based Question (Pseudo Visual Input)

In [19]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  The gallbladder is retracted superiorly, and dissection is being performed around Calot’s triangle. What is the current step, what is the next step, and what are the risks?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                        | 0/1 [00:00<?…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The dissection begins by incising peritoneum along the edge of the gallbladder on both sides to open up the hepatocystic triangle.",
    "current_step": "Dissecting Calot's triangle",
    "next_action": "Continue dissecting the triangle to expose the cystic duct and artery",
    "safety_warnings": "Avoid energy use near the duodenum which can be adherent to the gallbladder"
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### (Optional) Interactive UI Dashboard (`ipywidgets`)

As an alternative to the programmatic testing shown above, I have also engineered a lightweight, graphical User Interface directly within this Jupyter environment. 

While setting up a separate web framework is common in production, it adds extra network layers and delay, which aren’t needed for a local HPC demo.

To fulfill the UI requirement efficiently, this cell utilizes `ipywidgets` to generate a native, interactive dashboard. It allows us to dynamically test new clinical scenarios, ask follow-up reasoning questions, and evaluate the agent's intent-classification in real-time, completely bypassing the need to modify code or re-execute cells.

__`ipywidgets`__ allows you to build a sleek, interactive graphical interface directly inside the notebook cell.

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def Clinical_Agent():
    # ==========================================
    # 1. Define the UI Components
    # ==========================================
    header = widgets.HTML("<h3>⚕️ Laparoscopic Cholecystectomy AI Assistant</h3><p>Enter your clinical query below to search the SAGES and ERAS guidelines.</p>")

    query_input = widgets.Textarea(
        value='',
        placeholder='e.g., "What is the current step if the surgeon is dissecting Calot’s triangle?"',
        description='🧑‍⚕️ Query:',
        layout=widgets.Layout(width='90%', height='80px')
    )

    submit_button = widgets.Button(
        description=' Ask Assistant',
        button_style='primary', # Makes the button blue
        icon='stethoscope',     # Adds a medical icon
        layout=widgets.Layout(width='200px', margin='10px 0px 10px 100px')
    )

    output_area = widgets.Output(layout=widgets.Layout(border='1px solid #d3d3d3', padding='10px', width='90%'))

    # ==========================================
    # 2. Define the Execution Logic
    # ==========================================
    def on_submit_clicked(b):
        with output_area:
            # Clear the previous output before showing the new one
            clear_output(wait=True)
        
            user_query = query_input.value.strip()
            if not user_query:
                print("⚠️ Please enter a valid question.")
                return

            print("🤖 AI Assistant is classifying intent and searching clinical guidelines...")
            
            try:
                # Invoke your Intent-Aware RAG chain
                result = rag_chain.invoke(user_query)
            
                # Print the structured output beautifully
                print("\n✅ Response Generated:\n")
                print(json.dumps(result, indent=4))
            
            except Exception as e:
                print(f"\n❌ An error occurred: {e}")

    # ==========================================
    # 3. Link and Display the UI
    # ==========================================
    submit_button.on_click(on_submit_clicked)

    # Display the elements vertically
    ui_layout = widgets.VBox([header, query_input, submit_button, output_area])
    display(ui_layout)

# Phase 5 Evaluation Script (Using RAGAS)

Getting the ground_truths is widely considered the biggest bottleneck in deploying enterprise AI. It is referred to as the "Cold Start Problem" of RAG evaluation.

In a real clinical production setting, building `ground_truths` data usually follows two main approaches.

The first is the gold-standard manual approach. We work closely with experienced surgeons and ask them to write a set of complex questions, along with the exact answers they expect the system to provide. This process takes time and resources, but it provides a reliable reference for safety and quality.

The second approach is more scalable and uses synthetic data generation. I use an LLM to create the test set. For example, I can loop through our ChromaDB, feed sections of the SAGES guidelines into Llama-3, and instruct it to act like a medical professor creating exam questions: ‘Read this paragraph and generate three factual questions with their answers.’

This allows us to quickly build a large dataset of ground-truth QA pairs, which we can then use for evaluation with frameworks like RAGAS.

__Important Note:__

If your goal is to use RAGAS to explicitly evaluate the exact question you plan to type during the live demo, __you cannot use the synthetic generator__. The synthetic generator is only for building massive test suites.

If you want to evaluate one specific question, you must manually provide the __"Gold Standard"__ ground truth yourself.

When you run RAGAS with a synthetic dataset, you are performing Offline Evaluation.

You are not evaluating the specific question you want to ask during the live demo. Instead, you are testing the system itself. Think of it like unit testing in software engineering, or giving a student a final exam. You test the agent on 50 random, diverse questions (your synthetic dataset) to see how it performs in general.

If the agent scores a 0.95 Faithfulness across those 50 random test questions, you now have statistical confidence that its underlying architecture (the Retriever, the Prompt, the LLM) is mathematically sound.

__If you just run a standard `for` loop 50 times with a single search term, you will end up with 50 slightly different questions about the same paragraph. To get 50 *diverse* questions, we need to engineer a script that actively manages the vector search so it doesn't repeat itself.__

## 5.1 Synthetic Test Data Generation

Since I already set up a Pydantic schema and a LangChain pipeline for my main agent, we can use that exact same architecture to build a Synthetic Data Generator.

In [17]:
import json
import random
from tqdm import tqdm  # Gives us a nice progress bar in the notebook
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

### 5.1.1 Define the Pydantic Schema & Strict Prompt

__Important Note:__

One of the biggest hurdles in building reliable LLM pipelines is parser stability. The Llama-3 Instruct model was able to generate the correct JSON output, but it didn’t stop there. Because it is trained to behave like a polite conversational assistant due to RLHF alignment, it continued adding extra text, such as explanations and sign-offs ("Best regards, [Your Name]"), after the JSON.

This caused a problem for LangChain’s JsonOutputParser, which expects strictly valid JSON. When it encountered the additional conversational text, the parsing step failed.

To address this, I applied a technique called prompt bounding. I explicitly instructed the model to behave like a programmatic service rather than a chatbot, and to return only the JSON output with no extra text. By adding these strict constraints, I prevented the model from producing conversational filler and ensured reliable parsing.

In [18]:
print("🚀 Initializing 50-Question Synthetic Exam Generator...")

# We force the LLM to output a clean Question and Answer pair.
class QA_Pair(BaseModel):
    question: str = Field(description="A specific clinical question based strictly on the text.")
    ground_truth: str = Field(description="The exact, factual textbook answer found in the text.")

qa_parser = JsonOutputParser(pydantic_object=QA_Pair)

generator_prompt = PromptTemplate(
    template="""You are an expert surgical educator creating an exam. 
    Read the following clinical guideline chunk. 
    Generate ONE difficult, specific question based on this text, and provide the exact factual answer.
    Do not use outside knowledge. 
    
    CRITICAL INSTRUCTIONS:
    1. You MUST output ONLY a single, valid JSON object.
    2. DO NOT output any conversational text, greetings, or explanations before or after the JSON.
    3. DO NOT say "Here is the output".
    
    Clinical Guideline Text:
    {text_chunk}
    
    {format_instructions}
    """,
    input_variables=["text_chunk"],
    partial_variables={"format_instructions": qa_parser.get_format_instructions()},
)

# We reuse the `llm` (vLLM) already initialized in Phase 4
generation_chain = generator_prompt | llm | qa_parser

🚀 Initializing 50-Question Synthetic Exam Generator...


### 5.1.2 Define the Syllabus (Topic Bank)

In [19]:
# The script will randomly pull from these to ensure broad coverage
topic_bank = [
    "Patient positioning and operating room setup",
    "Trocar placement and pneumoperitoneum",
    "Identification of the cystic duct and artery",
    "Achieving the Critical View of Safety",
    "Management of bleeding during dissection",
    "Bile duct injury recognition and prevention",
    "Indications for converting to open cholecystectomy",
    "Postoperative pain management (ERAS protocols)",
    "Postoperative nausea and vomiting prophylaxis",
    "Discharge criteria and patient follow-up"
]

### 5.1.3 The Generation Loop

__Avoid "infinite while loop" trap when generating synthetic data:__

If your script is getting stuck at exactly 7/50, it means it has mathematically run out of fresh text chunks to read, so it is just spinning in circles forever.

__The Bottleneck: The Math of k=5__

In the script, if we gave the Retriever 10 topics and told it to pull the top 5 chunks for each topic (`k=5`).
10 topics × 5 chunks = 50 maximum possible chunks.

However, out of those 50 chunks, some are probably too short (less than 150 characters), and some probably triggered a parsing error when the LLM read them. Once the script burns through the handful of good chunks (ex, 7 of them), every time it searches ChromaDB, it just pulls the same chunks it has already used. The script sees they are in the `used_chunks` list, skips them, and loops infinitely.

To fix this, we need to do three things:
- __Increase `k`:__ Tell the Retriever to pull the top 30 or 50 chunks per topic instead of just 5.
- __Add an Escape Hatch:__ Implement a `max_attempts` counter so the loop eventually gives up instead of freezing your notebook.
- __Unsilence the Errors:__ We need to see if Llama-3 is secretly crashing in the background.

In [20]:
TARGET_QUESTIONS = 50
synthetic_dataset = []
used_chunks = set() 

# FIX 1: Add a safety counter to prevent infinite loops
max_attempts = 200 
attempts = 0

print(f"📚 Generating {TARGET_QUESTIONS} unique questions...\n")

with tqdm(total=TARGET_QUESTIONS, desc="Generating Exam") as pbar:
    
    while len(synthetic_dataset) < TARGET_QUESTIONS and attempts < max_attempts:
        attempts += 1
        
        current_topic = random.choice(topic_bank)
        
        # FIX 2: Dramatically increase k to cast a wider net
        candidate_docs = vector_store.similarity_search(current_topic, k=30)
        
        selected_doc = None
        for doc in candidate_docs:
            chunk_hash = hash(doc.page_content)
            
            if chunk_hash not in used_chunks and len(doc.page_content) > 150:
                selected_doc = doc.page_content
                used_chunks.add(chunk_hash)
                break 
                
        if not selected_doc:
            continue
            
        try:
            qa_result = generation_chain.invoke({"text_chunk": selected_doc})
            
            synthetic_dataset.append({
                "question": qa_result.get("question"),
                "answer": "",  
                "contexts": [selected_doc],
                "ground_truths": [qa_result.get("ground_truth")]
            })
            
            pbar.update(1) 
            
        except Exception as e:
            # FIX 3: Print a tiny warning so we know WHY it's failing
            # `\r` ensures it prints cleanly without breaking the tqdm progress bar
            tqdm.write(f"\r⚠️ Generation failed: {str(e)[:100]}...")

if len(synthetic_dataset) < TARGET_QUESTIONS:
    print(f"\n⚠️ Reached max attempts! Only found enough unique data for {len(synthetic_dataset)} questions.")
else:
    print(f"\n✅ Success! Generated all {TARGET_QUESTIONS} questions.")

📚 Generating 50 unique questions...



Generating Exam:   0%|                                                                                                          | 0/50 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                     | 0/1 [00:00<?, ?it/s, est. speed…

Generating Exam:   2%|█▉                                                                                                | 1/50 [00:26<21:15, 26.02s/it]

...

Generating Exam:  30%|█████████████████████████████                                                                    | 15/50 [04:25<10:52, 18.65s/it]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                     | 0/1 [00:00<?, ?it/s, est. speed…

Generating Exam:  32%|███████████████████████████████                                                                  | 16/50 [04:59<10:37, 18.74s/it]


⚠️ Reached max attempts! Only found enough unique data for 16 questions.


In [ ]:
synthetic_dataset

If your database is relatively small (e.g., just the single ERAS PDF and a few web pages), it might still stop before hitting 50, but it will do so gracefully because of the `max_attempts` cutoff. If it stops at, say, 25, that is completely fine! 25 unique, high-quality questions are more than enough to prove your evaluation framework.

### 5.1.4 Save the Dataset to Disk

In [22]:
output_filename = "synthetic_ragas_dataset.json"
with open(output_filename, "w") as f:
    json.dump(synthetic_dataset, f, indent=4)

print(f"\n✅ Success! Saved {TARGET_QUESTIONS} diverse QA pairs to '{output_filename}'.")


✅ Success! Saved 50 diverse QA pairs to 'synthetic_ragas_dataset.json'.


### 5.1.5 Load the Synthetic Ground Truths

In [20]:
dataset_filename = "synthetic_ragas_dataset.json"

with open(dataset_filename, "r") as f:
    synthetic_data = json.load(f)

print(f"📥 Successfully loaded {len(synthetic_data)} test questions from disk.")

📥 Successfully loaded 16 test questions from disk.


To evaluate the system, I needed a dataset of ground truths. Rather than manually copying and pasting from the ERAS PDF, I engineered an automated pipeline that inverses the RAG process. I used Llama-3 and LangChain to read the database and synthetically generate its own clinical exam. This output can be directly plugged into the RAGAS evaluation script.

## 5.2 Evaluation

Because the agent outputs dynamic JSON and free-text, you can't just use traditional machine learning metrics like F1-scores or standard accuracy.

If I were moving this pipeline into a real production environment, I would implement an evaluation framework like RAGAS (Retrieval Augmented Generation Assessment)

To truly know if the agent is performing well, you have to split the evaluation into two distinct parts:

- __Retrieval Evaluation:__ First, we have to measure if the ChromaDB is actually pulling the right clinical guidelines. We measure things like Context Precision (are the retrieved chunks highly relevant?) and Context Recall (did we miss any crucial steps from the ERAS manual?).
- __Generation Evaluation:__ Second, we evaluate the Llama-3 model’s output. The two most important metrics here are Faithfulness and Answer Relevance. Faithfulness checks that the model isn’t making things up, and that every claim in the JSON response is supported by the retrieved clinical context. Answer Relevance ensures the model is directly addressing the surgeon’s question, rather than drifting into unrelated details.

In a modern MLOps (Machine Learning Operations) pipeline, we evaluate these metrics using the ‘LLM-as-a-judge’ approach. We pass the user’s question, the retrieved context, and the model’s final JSON output to a more advanced model like GPT-4o or Claude 3.5 Sonnet, and ask it to score the response based on specific RAGAS metrics.
This allows us to run automated regression tests whenever we update the surgical guidelines or make small adjustments to the embedding model.

In [21]:
import json
from tqdm import tqdm
from datasets import Dataset
from ragas import evaluate
import pandas as pd
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

/scratch/f0034wq/ipykernel_2882350/2762644384.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2882350/2762644384.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2882350/2762644384.py:6: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/scratch/f0034wq/ipykernel_2882350/2762644384.py:6: DeprecationWarning: Importing context_r

The JSON I just saved contains the `question` and the `ground_truths` (from phase 5.1), but __it does not have the agent's answers yet__. Before I throw it into the evaluator, I must load the questions, feed them to my live Llama-3 agent (the Phase 4 `rag_chain`), record what the agent says, and then package it all up for RAGAS.

__Brief introduction:__

Evaluating an AI pipeline is a multi-step process. The synthetic JSON file serves as our fixed ‘exam set.’ It contains the questions along with the exact textbook answers. To evaluate the system, the agent needs to go through that exam.

I built an automated inference loop that reads the JSON file, extracts each question, and sends them one by one to our local Llama-3 agent. The script records the clinical context retrieved from ChromaDB, along with the final JSON response generated by the agent.

After all questions are processed, the script reorganizes the data into a structured Hugging Face Dataset. This allows RAGAS to compare the model’s answers with the textbook `ground_truth` and retrieved context, and generate a final evaluation report.

### 5.2.1 Setup the RAGAS Dictionary Structure

In [22]:
# Hugging Face Datasets require a dictionary of lists

# RAGAS requires 4 specific data points for every test case:
# - question: The user's query
# - answer: The actual text generated by your Llama-3 agent
# - contexts: The text chunks retrieved by ChromaDB
# - reference: The "perfect" answer (used to test recall)

eval_dict = {
    "question": [],
    "answer": [],
    "contexts": [], # The chunks returned by your retriever
    "reference": []
}

### 5.2.2 The Inference Loop (Testing the Agent)

In [ ]:
print("\n🤖 Running test questions through the Llama-3 Agent...")

for item in tqdm(synthetic_data, desc="Evaluating Agent"):
    
    test_question = item["question"]
    # Extract the textbook answer from our JSON
    ground_truth = item["ground_truths"]
    
    try:
        # A. Get the Contexts
        # We manually call the retriever so we can explicitly pass the text to RAGAS
        retrieved_docs = retriever.invoke(test_question)
        contexts = [doc.page_content for doc in retrieved_docs]
        
        # B. Get the Agent's Answer
        # We invoke your Phase 4 pipeline. 
        # (Assuming your chain expects a raw string and handles the dict internally)
        agent_response = rag_chain.invoke(test_question)
        
        # Extract the main text from your Pydantic output dictionary
        primary_answer = agent_response.get("primary_answer", "Error: No answer generated.")
        
        # C. Append everything to our evaluation dictionary
        eval_dict["question"].append(test_question)
        eval_dict["answer"].append(primary_answer)
        eval_dict["contexts"].append(contexts)

        # RAGAS 'reference' expects a single string.
        # Since our synthetic dataset saved it as a list, we extract the first item [0].
        eval_dict["reference"].append(ground_truth[0] if isinstance(ground_truth, list) else ground_truth)
        
    except Exception as e:
        tqdm.write(f"\r⚠️ Agent failed to answer question: {str(e)}")

### 5.2.3 Convert to Hugging Face Dataset & Evaluate

In [ ]:
OPENAI_API_KEY = "Your_Key"

In [33]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY # https://platform.openai.com/account/api-keys
assert os.environ.get("OPENAI_API_KEY") is not None, "Please set OPENAI_API_KEY environment variable"

In [ ]:
print("\n📊 Converting to Hugging Face Dataset and running RAGAS metrics...")

# Convert our Python dictionary into the format RAGAS requires (HuggingFace Dataset object)
hf_dataset = Dataset.from_dict(eval_dict)

# Run the LLM-as-a-judge evaluation!
print("Running RAGAS Evaluation Pipeline...")
# Note: In a production setting, you can configure RAGAS to use a strong local judge (like Llama-3 70B) or OpenAI's GPT-4o.
results = evaluate(
    dataset = hf_dataset, 
    metrics = [
        context_precision, # Did the retriever pull relevant docs?
        context_recall,    # Did the retriever miss any textbook facts?
        faithfulness,      # Did the agent hallucinate outside the context?
        answer_relevancy   # Did the agent actually answer the specific question?
    ]
)


#### 5.2.3.1 Alternatively Local Mode

By default, RAGAS is designed with OpenAI’s cloud infrastructure in mind. To speed up evaluation, it uses asynchronous execution to send many evaluation requests at the same time.

When we switch from OpenAI to a local Llama-3 model running on a single GPU, this behavior becomes a problem [DDOS (Distributed Denial of Service) attack]. RAGAS ends up sending a large number of requests in parallel to the local vLLM service. The GPU memory quickly fills up, the requests start to queue, and the system can freeze.

Another challenge is that RAGAS hides errors by default. If the local model runs into issues like running out of memory or timing out, those errors may not be shown, making it seem like the process is stuck.

To fix this, we need to clearly configure RAGAS for a local setup. I use `RunConfig` to limit the number of parallel requests to one, and I enable `raise_exceptions` so any issues are reported immediately. This makes the evaluation process stable and much easier to debug.

- __It will be slower:__ Because it is processing sequentially (`max_workers=1`), it might take a few minutes to grade all 50 questions. That is the necessary trade-off for running local AI. (If your HPC node is incredibly powerful, you can try bumping max_workers=2 or max_workers=4 to speed it up).

- __If it still crashes:__ Because we added `raise_exceptions=True`, it will no longer freeze silently. It will output a massive red error block telling you exactly why the local LLM failed (e.g., "CUDA Out of Memory" or "Context length exceeded"), which makes debugging infinitely easier.

In [24]:
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# Import the RAGAS wrappers for LangChain models
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

/scratch/f0034wq/ipykernel_2882350/2118320208.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/scratch/f0034wq/ipykernel_2882350/2118320208.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/scratch/f0034wq/ipykernel_2882350/2118320208.py:1: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_pre

In [ ]:
# Convert our Python dictionary into the format RAGAS requires (HuggingFace Dataset object)
hf_dataset = Dataset.from_dict(eval_dict)

# Wrap your existing local models for RAGAS
# Make sure 'llm' is your vLLM object and 'bge_embeddings' is your BGE object from Phase 3/4
ragas_local_llm = LangchainLLMWrapper(llm)
ragas_local_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("\n🧠 Running fully localized RAGAS evaluation (No OpenAI APIs)...")

# Configure RAGAS for a local GPU
# max_workers=1 forces it to evaluate one question at a time
local_config = RunConfig(
    max_workers=1, 
    timeout=120, 
    max_retries=2
)

# 3. Run the evaluation, explicitly overriding the OpenAI defaults
results = evaluate(
    dataset=hf_dataset, 
    metrics=[
        faithfulness, 
        answer_relevancy, 
        context_precision, 
        context_recall
    ],
    llm=ragas_local_llm,               # Forces RAGAS to use Llama-3 as the Judge
    embeddings=ragas_local_embeddings, # Forces RAGAS to use BGE-small for math
    run_config=local_config,
    raise_exceptions=True              # Force RAGAS to print the exact error if it crashes
)

### 5.2.4 Display the Final Scorecard

In [35]:
print("\n✅ Evaluation Complete! Here is the Agent's Report Card:")

df_results = results.to_pandas()
display(df_results)

# Optional: Print the average scores
print("\n🏆 Average System Performance:")
print(f"Faithfulness: {df_results['faithfulness'].mean():.4f}")
print(f"Answer Relevancy: {df_results['answer_relevancy'].mean():.4f}")
print(f"Context Precision: {df_results['context_precision'].mean():.4f}")
print(f"Context Recall: {df_results['context_recall'].mean():.4f}")


✅ Evaluation Complete! Here is the Agent's Report Card:


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,What is the basis for the use of the critical ...,[GUIDELINE RECOMMENDATIONS:\nQuestion 1: Shoul...,In patients undergoing laparoscopic cholecyste...,Two lines of indirect evidence: several large ...,1.0,0.0,1.0,NaN
1,What was the conversion rate in the early grou...,[Conversion to open cholecystectomy: This vari...,The conversion rate was 13.4% in the early gro...,13.4%,1.0,1.0,1.0,NaN
2,What is the recommendation for limiting the ri...,[Abstract: Laparoscopic cholecystectomy remain...,The recommendation for limiting the risk or se...,No recommendation was made,1.0,0.0,0.5,NaN
3,What is a common consideration when identifyin...,[or cutting of any structures. At this junctur...,One common consideration is ensuring the right...,ensuring the right hepatic artery is not mista...,1.0,1.0,1.0,NaN
4,What type of energy device is recommended for ...,[operation. For the difficult gallbladder in t...,For the difficult gallbladder in the setting o...,an advanced energy device such as an ultrasoni...,1.0,1.0,1.0,NaN
5,What is the definition of the cystic plate in ...,[The critical view of safety requires three cr...,The cystic plate is defined as the liver bed o...,The liver bed of the gallbladder and represent...,1.0,1.0,1.0,NaN
6,What type of data are at high risk of bias and...,[claims or administrative data that are at hig...,claims or administrative data that are at high...,claims or administrative data,1.0,1.0,1.0,NaN
7,What type of energy device may be used to main...,[operation. For the difficult gallbladder in t...,For the difficult gallbladder in the setting o...,an ultrasonic coagulator,1.0,1.0,1.0,NaN
8,When will you stop receiving antibiotics after...,[remain in the hospital for 3-5 days.\nNon-Sur...,A few weeks after your drain is placed you may...,As long as your gallbladder was not perforated...,0.0,0.0,1.0,NaN
9,How long should you avoid intense activity aft...,[activity without overdoing it and should not ...,Avoid intense activity for 10 to 14 days after...,10 to 14 days,1.0,1.0,1.0,NaN



🏆 Average System Performance:
Faithfulness: 0.8125
Answer Relevancy: nan
Context Precision: 0.7500
Context Recall: 0.6875


__Why got NaN in Answer Relevancy?__

RAGAS uses an approach to calculate answer relevancy. It takes the model’s generated answer, asks a judge LLM (GPT-4) to generate three possible questions that this answer could address, and then calculates the Cosine Similarity between them and the original user query using embedding distance (usually OpenAI's `text-embedding-ada-002`).

When you get a `NaN`, it means one of those steps returned a null or zero value, causing a divide-by-zero math error. Here are the three most likely reasons when using OpenAI:

1. __The "Empty Answer" Trap (Most Likely):__ Check the raw data you fed into RAGAS. During your inference loop, did your Llama-3 agent fail to generate a proper JSON response for any of the questions? If your extraction code (`agent_response.get("primary_answer", "")`) passed an empty string, or an error string like `"Error: No answer generated"`, GPT-4 literally cannot reverse-engineer a question from it. Furthermore, the embedding model cannot calculate the mathematical distance of an empty concept. RAGAS will silently fail on that row and output `NaN` for the average.

2. __OpenAI's Content Moderation Filters:__ You are building a surgical AI. Your text is full of phrases like "dissection," "bleeding," "excision," and "injury." OpenAI has very strict, automated safety filters. Sometimes, when RAGAS sends a batch of surgical text to the GPT-4 API, the OpenAI moderation layer flags it as "violence" or "self-harm" and blocks the API request. When the API request is blocked, RAGAS receives an empty response instead of the reverse-engineered questions, which breaks the math and results in `NaN`.

3. __API Rate Limits (Concurrency):__ RAGAS evaluates asynchronously. It fires off dozens of requests to the OpenAI API simultaneously. If you are using a standard Tier 1 or Tier 2 OpenAI developer account, you might have hit your Tokens-Per-Minute (TPM) or Requests-Per-Minute (RPM) limit. GPT-4 will return an HTTP 429 Error ("Too Many Requests"). If RAGAS suppresses this error, it logs the score as a null value, dragging your final average to `NaN`.


__Context Precision: 0.7500 (Strong):__
When the Retriever pulls documents from ChromaDB, it successfully puts the most relevant, highly-ranked chunks at the very top of the context window 75% of the time. Your MMR (Maximal Marginal Relevance) search configuration and BGE-small embeddings are working very well.

__Context Recall: 0.6875 (The Bottleneck):__
The Retriever is missing about 31% of the necessary textbook facts needed to fully answer the questions. This is the weakest link in your pipeline right now. It usually means your `chunk_size` is too small, or your `k` value (how many chunks you retrieve) is too low. The agent simply isn't being fed the complete picture.

__Faithfulness: 0.8125 (Good, but clinically risky):__
Roughly 81% of the claims your agent makes are perfectly backed up by the textbook. However, almost 20% of the time, the model provides incorrect information that isn't in the provided context. In an e-commerce chatbot, 81% would be considered strong performance. In a surgical setting, though, that level of accuracy introduces significant risk. This is likely driven by the lower Context Recall. If the retriever is missing about 31% of the relevant information, the LLM still tries to provide a complete answer and ends up filling in gaps using its pre-trained knowledge. RAGAS then identifies those unsupported statements as errors.

#### 5.2.4.1 Display the Final Scorecard (using local model for evaluation)

In [27]:
print("\n✅ Evaluation Complete! Here is the Agent's Report Card:")

df_results = results.to_pandas()
display(df_results)

# Optional: Print the average scores
print("\n🏆 Average System Performance (local model version):")
print(f"Faithfulness: {df_results['faithfulness'].mean():.4f}")
print(f"Answer Relevancy: {df_results['answer_relevancy'].mean():.4f}")
print(f"Context Precision: {df_results['context_precision'].mean():.4f}")
print(f"Context Recall: {df_results['context_recall'].mean():.4f}")


✅ Evaluation Complete! Here is the Agent's Report Card:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the basis for the use of the critical ...,[GUIDELINE RECOMMENDATIONS:\nQuestion 1: Shoul...,In patients undergoing laparoscopic cholecyste...,Two lines of indirect evidence: several large ...,1.0,0.604073,1.000000,0.0
1,What was the conversion rate in the early grou...,[Conversion to open cholecystectomy: This vari...,The conversion rate was 13.4% in the early gro...,13.4%,0.5,1.000000,1.000000,1.0
2,What is the recommendation for limiting the ri...,[Abstract: Laparoscopic cholecystectomy remain...,The recommendation for limiting the risk or se...,No recommendation was made,0.5,0.960723,0.000000,0.0
3,What is a common consideration when identifyin...,[or cutting of any structures. At this junctur...,One common consideration is ensuring the right...,ensuring the right hepatic artery is not mista...,1.0,0.650367,1.000000,1.0
4,What type of energy device is recommended for ...,[operation. For the difficult gallbladder in t...,For the difficult gallbladder in the setting o...,an advanced energy device such as an ultrasoni...,1.0,0.718632,1.000000,1.0
5,What is the definition of the cystic plate in ...,[The critical view of safety requires three cr...,The cystic plate is defined as the liver bed o...,The liver bed of the gallbladder and represent...,1.0,0.863687,1.000000,1.0
6,What type of data are at high risk of bias and...,[claims or administrative data that are at hig...,claims or administrative data that are at high...,claims or administrative data,0.0,0.844636,1.000000,1.0
7,What type of energy device may be used to main...,[operation. For the difficult gallbladder in t...,For the difficult gallbladder in the setting o...,an ultrasonic coagulator,1.0,0.915636,1.000000,1.0
8,When will you stop receiving antibiotics after...,[remain in the hospital for 3-5 days.\nNon-Sur...,A few weeks after your drain is placed you may...,As long as your gallbladder was not perforated...,1.0,0.751242,0.000000,0.0
9,How long should you avoid intense activity aft...,[activity without overdoing it and should not ...,Avoid intense activity for 10 to 14 days after...,10 to 14 days,1.0,0.831807,1.000000,1.0



🏆 Average System Performance (local model versioin):
Faithfulness: 0.6875
Answer Relevancy: 0.8600
Context Precision: 0.7760
Context Recall: 0.6875


__Answer Relevancy 0.8600:__
The Llama-3 agent is very consistent in its behavior. When asked a specific surgical question, it directly answers that question about 86% of the time, rather than drifting into unrelated topics like hospital administration or other procedures.

__Context Precision 0.7760 (Strong):__
The embedding model (BGE-small) and ChromaDB are doing a great job. When they find relevant documents, they are successfully ranking them at the very top of the context window.

__Context Recall 0.6875 (The Bottleneck):__
This score clearly points to the main issue in the pipeline. The retriever is missing about 31% of the key facts needed to answer the questions. To address this, we can increase the chunk overlap and raise the `fetch_k` parameter so the system considers a broader set of candidate documents.

__Faithfulness 0.6875 (The Danger Zone):__
About 68% of the agent’s statements are supported by the textbook, which means that in roughly 32% of cases, it’s introducing information that isn’t supported by the source. In a surgical setting, that’s a serious concern.

This score was previously 81% when GPT-4 was used as the judge. The drop to 68% with Llama-3 is a known issue called judge misalignment. Smaller models, like an 8B parameter model, can struggle with subtle differences in wording.

For example, if the source says ‘the surgeon removed the gallbladder’ and the agent says ‘the gallbladder was excised,’ a smaller judge model may incorrectly mark that as unsupported, even though the meaning is the same. Larger models like GPT-4 are better at recognizing these equivalences.

__Summary:__

First, our Context Recall is at 0.68. Because the system is missing about 31% of the relevant context, the agent doesn’t have enough information and starts relying on its pre-trained knowledge to fill in the gaps.

Second, we’re seeing *Judge Misalignment*. Using an 8B model to evaluate another 8B model can lead to overly strict wording checks. It may mark equivalent clinical terms as incorrect because it doesn’t capture deeper meaning, as well as a larger model like GPT-4.

For the next version, I would increase the `k` parameter in vector search to improve recall, and use a larger model, such as a 70B model, as the local evaluator to get a more accurate and context-aware Faithfulness score.

### 5.2.5 Save as a CSV

In [36]:
output_csv_path = "ragas_evaluation_results_OpenAI.csv"

# The index=False parameter ensures Pandas doesn't write the arbitrary row numbers as a column
df_results.to_csv(output_csv_path, index=False)

print(f"💾 Successfully saved evaluation metrics to: {output_csv_path}")

💾 Successfully saved evaluation metrics to: ragas_evaluation_results_OpenAI.csv


In [28]:
output_csv_path = "ragas_evaluation_results_local_Llama.csv"

# The index=False parameter ensures Pandas doesn't write the arbitrary row numbers as a column
df_results.to_csv(output_csv_path, index=False)

print(f"💾 Successfully saved evaluation metrics to: {output_csv_path}")

💾 Successfully saved evaluation metrics to: ragas_evaluation_results_local_Llama.csv


Once the system passes the Offline Evaluation, we move to Runtime Inference.

This is the live demo. Because we already proved the system is mathematically sound using the synthetic dataset, we can now confidently type in any new question—like "What is the current step if the surgeon is dissecting Calot’s triangle?", and trust that the agent will give you a faithful, accurate answer. We do not run RAGAS during the live demo.